# Ridge — Rank-Y + Per-Target + Weighted Ensemble

## Evaluation Protocol

This notebook follows the offline evaluation guidelines for the MITSUI Commodity Prediction Challenge.  
Self-contained: runs without variables from other notebooks.

---

### X Feature Data

| Window | date_id | Source | Variable | Purpose |
|---|---|---|---|---|
| Train | 20–1568 | train.csv / X_train_lag1.csv | `X_train_c` | Fit all base models (A–E) |
| Validation | 1569–1826 | test.csv rows 0–257 | `X_val_c[:258]` | Select ensemble weights — training-period validation |
| Final Test | 1827–1960 | test.csv rows 258–391 | `X_val_c[258:]` | Final offline prediction only |

> `X_train_c` and `X_val_c` are the cleaned, zero-filled feature matrices (2850 columns after dropping high-missing / zero-variance columns from the training window). Lag-summary augmentation (`add_lag_summary_features`) is applied to both but produces no new columns in this dataset because the underlying `label_target_*_lag*` columns are all-zero in the training window — so augmented and non-augmented variants are numerically identical.

---

### Y Label Data

| Window | date_id | Source | Variable | Purpose |
|---|---|---|---|---|
| Validation | 1569–1826 | train_labels.csv (aligned, NaN preserved) | `val_solution_raw` | Validation scoring — used for ensemble weight selection only |
| Final Test | 1827–1960 | Reconstructed from test_labels_lag_1~4.csv, aligned by `label_date_id` | `test_ground_truth` | Final offline evaluation only — never used for model selection or tuning |

---

### Model Settings

| Model | Algorithm | Features | Y Transform | Hyperparameter |
|---|---|---|---|---|
| `A_no_aug` | Ridge (shared, all targets) | All 2850 base cols | Daily cross-sectional rank → [−1, 1] | α tuned by CV (TimeSeriesSplit, Spearman) |
| `A_aug` | Ridge (shared, all targets) | All 2850 + lag-summary | Daily cross-sectional rank → [−1, 1] | α tuned by CV |
| `B_raw` | Ridge (per-target, top-50) | Top-50 cols by train correlation | Raw (unranked) | α tuned by CV per target |
| `B_rank` | Ridge (per-target, top-50) | Top-50 cols by train correlation | Daily cross-sectional rank | α tuned by CV per target |
| `C_conservative` | Ridge (shared, all targets) | All 2850 base cols | Daily cross-sectional rank | Fixed α = 1 × 10⁴ (high regularisation) |
| `C_pca50` | PCA(50) + Ridge | 50 PCA components from 2850 cols | Daily cross-sectional rank | α tuned by CV on PCA space |
| `D_momentum` | Persistence (no training) | `label_target_*_lag*` from API / train_labels | — | Predicts last known return (lag offset = lag_k + 1) |
| `E_hgbr` | HistGradientBoosting (shared) | All 2850 base cols | Daily cross-sectional rank | max_iter=80, max_leaf_nodes=15, l2=1.0 |

---

**Final model: `ensemble_equal`** — pre-committed before any test scoring.  
Equal-weight average of all 8 candidates: A_no_aug, A_aug, B_raw, B_rank, C_conservative, C_pca50, D_momentum, E_hgbr.

### Leakage Avoidance
- All model fits use only `X_train_c` (date_id 20–1568).
- Ensemble weights are selected on `val_solution_raw` (date_id 1569–1826) only.
- `test_ground_truth` is loaded but untouched until Step 7 (final offline scoring).
- `D_momentum` val signal uses `train_labels.csv` shifted back by (lag_k + 1) rows — no future labels.

### How Test Labels Were Reconstructed
`test_ground_truth.csv` was reconstructed from `test_labels_lag_1.csv` through `test_labels_lag_4.csv`,  
aligned by `label_date_id` (not `date_id`), covering all 134 available test dates (1827–1960).


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from itertools import product

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR     = "/Users/ankit/Downloads/8. Predictive Project"
DATA_DIR     = f"{BASE_DIR}/1. After-Cleaned Dataset"
RAW_DATA_DIR = f"{BASE_DIR}/0. Dataset"
OUT_DIR3     = f"{BASE_DIR}/3. All Results from Ridge AllLags"   # notebook 3 output
OUT_DIR      = f"{BASE_DIR}/4. All Results from Ridge RankY"
os.makedirs(OUT_DIR, exist_ok=True)

ALPHA_GRID  = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5]
N_SPLITS_CV = 5
TOP_N_PER_TARGET = 50   # features per target in Exp B

LAG_GROUPS = {
    1: list(range(0,   106)),
    2: list(range(106, 212)),
    3: list(range(212, 318)),
    4: list(range(318, 424)),
}
ALL_TARGET_COLS = [f"target_{i}" for i in range(424)]

print("Output dir:", OUT_DIR)

Output dir: /Users/ankit/Downloads/8. Predictive Project/4. All Results from Ridge RankY


## Step 1 — Load & clean X

In [2]:
X_train_raw = pd.read_csv(f"{DATA_DIR}/X_train_lag1.csv")
X_val_raw   = pd.read_csv(f"{DATA_DIR}/X_val_lag1.csv")
assert X_train_raw.columns.equals(X_val_raw.columns)

# Column cleaning — identical to notebooks 2 & 3 (train-only rules)
miss_ratio        = X_train_raw.isna().mean()
high_missing_cols = miss_ratio[miss_ratio > 0.95].index.tolist()
X_train_f0 = X_train_raw.fillna(0.0)
X_val_f0   = X_val_raw.fillna(0.0)
low_var_cols = X_train_f0.var(axis=0)
low_var_cols = low_var_cols[low_var_cols < 1e-10].index.tolist()
drop_cols = sorted(set(high_missing_cols) | set(low_var_cols))
keep_cols = [c for c in X_train_raw.columns if c not in drop_cols]

X_train_c = X_train_f0[keep_cols].copy()
X_val_c   = X_val_f0[keep_cols].copy()
n_train       = X_train_c.shape[0]   # 1549
n_val         = X_val_c.shape[0]     # 392
VAL_TEST_SPLIT = 258  # rows 0-257 = validation (1569–1826), rows 258-391 = final test (1827–1960)

print(f"X_train_c: {X_train_c.shape}  X_val_c: {X_val_c.shape}")

X_train_c: (1549, 2850)  X_val_c: (392, 2850)


## Step 2 — Add lag-summary features

For each target i with `label_target_{i}_lag{1-4}` columns already in X,
add 5 summary statistics Ridge cannot learn on its own:
- mean, std, decay-weighted mean (recency bias), last-minus-mean (reversal), sign-consistency

In [3]:
def add_lag_summary_features(X_df):
    """
    Appends lag-summary columns for every target that has label_target_*_lag{1-4}.
    Fit on the input df itself — call with train first, then reuse col list for val.
    Returns (augmented_df, list_of_new_col_names).
    """
    # Find all target ids that have at least 2 lag columns
    target_ids = set()
    for c in X_df.columns:
        if c.startswith("label_target_") and "_lag" in c:
            try:
                tid = int(c.replace("label_target_", "").split("_lag")[0])
                target_ids.add(tid)
            except ValueError:
                pass

    new_cols = {}
    DECAY_W  = np.array([4.0, 3.0, 2.0, 1.0])  # lag1 most recent → highest weight

    for i in sorted(target_ids):
        lag_cols = [f"label_target_{i}_lag{k}" for k in [1, 2, 3, 4]
                    if f"label_target_{i}_lag{k}" in X_df.columns]
        if len(lag_cols) < 2:
            continue
        vals = X_df[lag_cols].values.astype(float)   # (n, n_lags)
        n_l  = vals.shape[1]
        w    = DECAY_W[:n_l] / DECAY_W[:n_l].sum()

        new_cols[f"lsf_{i}_mean"]        = vals.mean(axis=1)
        new_cols[f"lsf_{i}_std"]         = vals.std(axis=1, ddof=0)
        new_cols[f"lsf_{i}_decay"]       = vals @ w
        new_cols[f"lsf_{i}_lastmean"]    = vals[:, 0] - vals.mean(axis=1)
        new_cols[f"lsf_{i}_signconsist"] = (np.sign(vals) == np.sign(vals[:, 0:1])).mean(axis=1)

    if not new_cols:
        return X_df, []
    new_df     = pd.DataFrame(new_cols, index=X_df.index)
    new_col_names = list(new_cols.keys())
    return pd.concat([X_df, new_df], axis=1), new_col_names


X_train_aug, new_feature_names = add_lag_summary_features(X_train_c)
# Val: append same new columns (recomputed from val's own lag cols — no leakage)
X_val_aug, _ = add_lag_summary_features(X_val_c)

print(f"Added {len(new_feature_names)} lag-summary features")
print(f"X_train_aug: {X_train_aug.shape}  X_val_aug: {X_val_aug.shape}")

Added 0 lag-summary features
X_train_aug: (1549, 2850)  X_val_aug: (392, 2850)


## Step 3 — Load Y (no ffill) + obs masks

In [4]:
Y_train_lag1 = pd.read_csv(f"{DATA_DIR}/Y_train_lag1.csv")
Y_val_lag1   = pd.read_csv(f"{DATA_DIR}/Y_val_lag1.csv")
obs_val_lag1 = pd.read_csv(f"{DATA_DIR}/obs_val_lag1.csv")
obs_val_lag1 = obs_val_lag1[[f"target_{i}_observed" for i in LAG_GROUPS[1]]]

# Obs mask for TRAIN (needed for rank-Y transformation)
train_obs_cols = [f"target_{i}_observed" for i in range(424)
                  if f"target_{i}_observed" in X_train_raw.columns]
obs_train_all  = X_train_raw[train_obs_cols].fillna(1).reset_index(drop=True)
val_obs_all    = X_val_raw[[f"target_{i}_observed" for i in range(424)
                             if f"target_{i}_observed" in X_val_raw.columns]].fillna(1).reset_index(drop=True)

# Load & clean all Y (no ffill)
train_labels_raw = pd.read_csv(f"{RAW_DATA_DIR}/train_labels.csv")
train_labels_raw = train_labels_raw.sort_values("date_id").reset_index(drop=True)
target_cols      = [c for c in train_labels_raw.columns if c.startswith("target_")]

train_labels_clean = train_labels_raw.copy()
train_labels_clean[target_cols] = train_labels_clean[target_cols].fillna(0.0)  # no ffill

# Row alignment
lag1_ref = [f"target_{i}" for i in LAG_GROUPS[1]]
Y_ref    = Y_train_lag1.values
best_offset, best_mae = 0, np.inf
for off in range(0, train_labels_clean.shape[0] - n_train - n_val + 1):
    sub  = train_labels_clean[lag1_ref].iloc[off : off + n_train].values
    mask = (sub != 0) & (Y_ref != 0)
    if mask.sum() < 1000:
        continue
    mae = np.abs(sub[mask] - Y_ref[mask]).mean()
    if mae < best_mae:
        best_mae, best_offset = mae, off
print(f"Row alignment offset={best_offset}  MAE={best_mae:.2e}")

aligned = train_labels_clean.iloc[best_offset : best_offset + n_train + n_val].reset_index(drop=True)

Y_dfs = {}
for lag_num, indices in LAG_GROUPS.items():
    cols = [f"target_{i}" for i in indices]
    Y_dfs[lag_num] = {
        "train": aligned[cols].iloc[:n_train].reset_index(drop=True),
        "val":   aligned[cols].iloc[n_train:].reset_index(drop=True),
    }
Y_dfs[1]["train"] = Y_train_lag1
Y_dfs[1]["val"]   = Y_val_lag1

obs_dfs = {1: obs_val_lag1}
for lag_num, indices in LAG_GROUPS.items():
    if lag_num == 1:
        continue
    obs_cols = [f"target_{i}_observed" for i in indices]
    missing  = [c for c in obs_cols if c not in X_val_raw.columns]
    obs_dfs[lag_num] = X_val_raw[obs_cols].reset_index(drop=True) if not missing else \
        pd.DataFrame(np.ones((n_val, len(obs_cols)), dtype=int), columns=obs_cols)

# Val solution (NaN-preserving) for official scoring
# val_solution_raw: validation window only (date_id 1569–1826, 258 rows)
# Used ONLY for ensemble weight selection — never for final model choice
val_solution_raw = (
    train_labels_raw
    .iloc[best_offset + n_train : best_offset + n_train + VAL_TEST_SPLIT]
    [ALL_TARGET_COLS]
    .reset_index(drop=True)
)
print(f"val_solution_raw: {val_solution_raw.shape}  "
      f"NaN(closed): {val_solution_raw.isna().mean().mean()*100:.1f}%")

# test_ground_truth: final offline test labels (date_id 1827–1960, 134 rows)
# Reconstructed from test_labels_lag_1~4.csv aligned by label_date_id
# Used ONLY in Step 5 for final offline evaluation — NOT for tuning or model selection
_gt_raw = pd.read_csv(f"{BASE_DIR}/test_ground_truth.csv")
test_ground_truth = _gt_raw[ALL_TARGET_COLS].reset_index(drop=True)
print(f"test_ground_truth: {test_ground_truth.shape}  "
      f"NaN(closed): {test_ground_truth.isna().mean().mean()*100:.1f}%")

Row alignment offset=20  MAE=8.61e-27
val_solution_raw: (258, 424)  NaN(closed): 10.5%
test_ground_truth: (134, 424)  NaN(closed): 10.1%


## Step 4 — Core helper functions

In [5]:
# Official metric
def rank_correlation_sharpe_ratio(merged_df):
    prediction_cols = [c for c in merged_df.columns if c.startswith('prediction_')]
    target_cols_    = [c for c in merged_df.columns if c.startswith('target_')]

    def _row_rc(row):
        non_null = [c for c in target_cols_ if not pd.isnull(row[c])]
        preds    = [c for c in prediction_cols if c.replace('prediction', 'target') in non_null]
        if len(non_null) < 2:
            return np.nan
        if row[non_null].std(ddof=0) == 0 or row[preds].std(ddof=0) == 0:
            return np.nan
        return np.corrcoef(
            row[preds].rank(method='average'),
            row[non_null].rank(method='average')
        )[0, 1]

    daily = merged_df.apply(_row_rc, axis=1).dropna()
    std   = daily.std(ddof=0)
    return float(daily.mean() / std) if std > 0 else 0.0


def official_score(pred_df_424, val_solution):
    """Score a (392, 424) prediction DataFrame against val_solution."""
    sub    = pred_df_424.rename(columns={c: c.replace('target_', 'prediction_')
                                          for c in pred_df_424.columns})
    merged = pd.concat([val_solution.copy(), sub], axis=1)
    return rank_correlation_sharpe_ratio(merged)


def build_full_pred_df(lag_preds):
    """Combine {lag_num: (392,106) array} into sorted (392,424) DataFrame."""
    parts = []
    for lag_num, indices in sorted(LAG_GROUPS.items()):
        cols = [f"target_{i}" for i in indices]
        parts.append(pd.DataFrame(lag_preds[lag_num], columns=cols))
    return pd.concat(parts, axis=1)[ALL_TARGET_COLS]

In [6]:
def rank_transform_Y(Y_df, obs_train_df=None):
    """
    For each day, rank observed targets' returns and map to [-1, 1].
    Unobserved targets (obs==0) get 0 (neutral prediction).
    """
    Y_arr = Y_df.values.copy().astype(float)
    Y_rank = np.zeros_like(Y_arr)
    target_names = list(Y_df.columns)

    for t in range(Y_arr.shape[0]):
        if obs_train_df is not None:
            obs_cols_here = [f"{c}_observed" for c in target_names
                             if f"{c}_observed" in obs_train_df.columns]
            if obs_cols_here:
                obs_row = obs_train_df[obs_cols_here].iloc[t].values.astype(bool)
            else:
                obs_row = np.ones(Y_arr.shape[1], dtype=bool)
        else:
            obs_row = (Y_arr[t] != 0)   # fallback: treat 0-fill as closed

        obs_idx = np.where(obs_row)[0]
        if len(obs_idx) < 2:
            continue
        vals  = Y_arr[t, obs_idx]
        ranks = pd.Series(vals).rank(method='average').values
        n     = len(ranks)
        Y_rank[t, obs_idx] = 2 * (ranks - 1) / (n - 1) - 1   # [-1, 1]

    return pd.DataFrame(Y_rank, columns=Y_df.columns)


def cv_spearman(X_s, Y_s, alpha, n_splits=N_SPLITS_CV):
    """CV mean Spearman for one alpha (in scaled-Y space)."""
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr, vl in tscv.split(X_s):
        m    = Ridge(alpha=alpha, random_state=42)
        m.fit(X_s[tr], Y_s[tr])
        pred = m.predict(X_s[vl])
        fold_sp = []
        for j in range(Y_s.shape[1]):
            yt, yp = Y_s[vl, j], pred[:, j]
            if np.std(yt) > 1e-12 and np.std(yp) > 1e-12:
                fold_sp.append(float(spearmanr(yt, yp).correlation))
        if fold_sp:
            scores.append(float(np.mean(fold_sp)))
    return float(np.mean(scores)) if scores else -np.inf


def tune_alpha(X_s, Y_s, alpha_grid=ALPHA_GRID):
    best_a, best_sc = alpha_grid[0], -np.inf
    for a in alpha_grid:
        sc = cv_spearman(X_s, Y_s, a)
        if sc > best_sc:
            best_sc, best_a = sc, a
    return best_a

## Experiment A — Rank-transformed Y Ridge

Train Ridge on daily cross-sectional rank-mapped Y instead of raw returns.
Directly aligns training with the official ranking objective.

In [7]:
def run_exp_a(X_tr_c, X_vl_c, Y_tr, Y_vl, obs_tr_all, lag_num, use_aug=False):
    """
    Experiment A: rank-transform Y daily, then Ridge.
    use_aug: if True, use X with lag-summary features appended.
    """
    X_tr = X_train_aug if use_aug else X_tr_c
    X_vl = X_val_aug   if use_aug else X_vl_c

    # Build obs mask for train targets of this lag
    tgt_names = list(Y_tr.columns)
    obs_tr_cols = [f"{c}_observed" for c in tgt_names
                   if f"{c}_observed" in obs_tr_all.columns]
    obs_tr_lag = obs_tr_all[obs_tr_cols] if obs_tr_cols else None

    # Rank-transform Y
    Y_tr_rank = rank_transform_Y(Y_tr, obs_tr_lag)

    scaler_X = StandardScaler()
    X_tr_s   = scaler_X.fit_transform(X_tr.values)
    X_vl_s   = scaler_X.transform(X_vl.values)

    # Y is already [-1,1] — still scale for Ridge stability
    scaler_Y = StandardScaler()
    Y_tr_s   = scaler_Y.fit_transform(Y_tr_rank.values)

    tag = f"A_lag{lag_num}" + ("_aug" if use_aug else "")
    print(f"  [{tag}] features={X_tr_s.shape[1]}  tuning alpha...")
    best_alpha = tune_alpha(X_tr_s, Y_tr_s)
    print(f"  [{tag}] best_alpha={best_alpha}")

    model    = Ridge(alpha=best_alpha, random_state=42)
    model.fit(X_tr_s, Y_tr_s)
    pred_s   = model.predict(X_vl_s)
    pred     = scaler_Y.inverse_transform(pred_s)

    return pred   # (n_val, 106) — raw predictions


print("Running Experiment A (rank-Y Ridge) for all 4 lags...")
exp_a_preds     = {}   # {lag_num: array (392, 106)}
exp_a_aug_preds = {}   # with lag-summary features

for lag_num in [1, 2, 3, 4]:
    print(f"\n--- Lag {lag_num} ---")
    Y_tr   = Y_dfs[lag_num]["train"]
    Y_vl   = Y_dfs[lag_num]["val"]

    exp_a_preds[lag_num]     = run_exp_a(X_train_c, X_val_c, Y_tr, Y_vl,
                                          obs_train_all, lag_num, use_aug=False)
    exp_a_aug_preds[lag_num] = run_exp_a(X_train_c, X_val_c, Y_tr, Y_vl,
                                          obs_train_all, lag_num, use_aug=True)

score_a     = official_score(build_full_pred_df(exp_a_preds),     val_solution_raw)
score_a_aug = official_score(build_full_pred_df(exp_a_aug_preds), val_solution_raw)
print(f"\nExp A (rank-Y, no aug):       {score_a:.6f}")
print(f"Exp A (rank-Y, +lag-summary): {score_a_aug:.6f}")

Running Experiment A (rank-Y Ridge) for all 4 lags...

--- Lag 1 ---
  [A_lag1] features=2850  tuning alpha...
  [A_lag1] best_alpha=1000.0
  [A_lag1_aug] features=2850  tuning alpha...
  [A_lag1_aug] best_alpha=1000.0

--- Lag 2 ---
  [A_lag2] features=2850  tuning alpha...
  [A_lag2] best_alpha=1000.0
  [A_lag2_aug] features=2850  tuning alpha...
  [A_lag2_aug] best_alpha=1000.0

--- Lag 3 ---
  [A_lag3] features=2850  tuning alpha...
  [A_lag3] best_alpha=1000.0
  [A_lag3_aug] features=2850  tuning alpha...
  [A_lag3_aug] best_alpha=1000.0

--- Lag 4 ---
  [A_lag4] features=2850  tuning alpha...
  [A_lag4] best_alpha=1000.0
  [A_lag4_aug] features=2850  tuning alpha...
  [A_lag4_aug] best_alpha=1000.0

Exp A (rank-Y, no aug):       0.185347
Exp A (rank-Y, +lag-summary): 0.185347


## Experiment B — Per-target top-50 Ridge

Each of 424 targets gets its own top-50 feature subset and alpha.
Eliminates noise from the shared 2850-col pool.
Try on both raw Y and rank-Y.

In [9]:
def run_per_target_ridge(X_tr, X_vl, Y_tr, top_n=TOP_N_PER_TARGET,
                          alpha_grid=ALPHA_GRID, label=""):
    """
    For each target column in Y_tr:
      1. Select top-N features by |Pearson corr| with that target (train only)
      2. Tune alpha by CV Spearman
      3. Fit Ridge on full train
    Returns prediction array (n_val, n_targets).
    """
    Xv      = X_tr.values
    X_c     = Xv - Xv.mean(axis=0, keepdims=True)
    x_std   = X_c.std(axis=0, ddof=0) + 1e-12
    n       = Xv.shape[0]

    n_tgts  = Y_tr.shape[1]
    preds   = np.zeros((X_vl.shape[0], n_tgts))

    for j, tgt in enumerate(Y_tr.columns):
        y   = Y_tr[tgt].values
        y_c = y - y.mean()
        y_s = y_c.std(ddof=0) + 1e-12

        # Correlation of every feature with this target
        corr   = (X_c.T @ y_c) / (x_std * y_s * n)
        top_idx = np.argsort(-np.abs(corr))[:top_n]
        sel_cols = [X_tr.columns[i] for i in top_idx]

        X_tr_sel = X_tr[sel_cols].values
        X_vl_sel = X_vl[sel_cols].values

        scaler_X = StandardScaler()
        X_tr_s   = scaler_X.fit_transform(X_tr_sel)
        X_vl_s   = scaler_X.transform(X_vl_sel)

        scaler_y = StandardScaler()
        y_s_arr  = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

        # Alpha tuning: 1D Spearman CV (fast)
        best_a, best_sc = alpha_grid[0], -np.inf
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)
        for a in alpha_grid:
            fold_sps = []
            for tr_idx, vl_idx in tscv.split(X_tr_s):
                m = Ridge(alpha=a, random_state=42)
                m.fit(X_tr_s[tr_idx], y_s_arr[tr_idx])
                p = m.predict(X_tr_s[vl_idx])
                if np.std(y_s_arr[vl_idx]) > 1e-12 and np.std(p) > 1e-12:
                    fold_sps.append(float(spearmanr(y_s_arr[vl_idx], p).correlation))
            sc = float(np.mean(fold_sps)) if fold_sps else -np.inf
            if sc > best_sc:
                best_sc, best_a = sc, a

        model = Ridge(alpha=best_a, random_state=42)
        model.fit(X_tr_s, y_s_arr)
        pred_s      = model.predict(X_vl_s)
        preds[:, j] = scaler_y.inverse_transform(pred_s.reshape(-1, 1)).ravel()

        if j % 50 == 0:
            print(f"    {label} target {j}/{n_tgts}  alpha={best_a}")

    return preds


print("Running Experiment B (per-target top-50) for all 4 lags...")
exp_b_raw_preds  = {}   # raw Y
exp_b_rank_preds = {}   # rank-Y

for lag_num in [1, 2, 3, 4]:
    print(f"\n--- Lag {lag_num} ---")
    Y_tr    = Y_dfs[lag_num]["train"]

    # B1: raw Y
    print(f"  [B_raw_lag{lag_num}] per-target top-{TOP_N_PER_TARGET}...")
    exp_b_raw_preds[lag_num] = run_per_target_ridge(
        X_train_aug, X_val_aug, Y_tr,
        label=f"B_raw_lag{lag_num}"
    )

    # B2: rank-Y (get obs for this lag)
    tgt_names   = list(Y_tr.columns)
    obs_tr_cols = [f"{c}_observed" for c in tgt_names
                   if f"{c}_observed" in obs_train_all.columns]
    obs_tr_lag  = obs_train_all[obs_tr_cols] if obs_tr_cols else None
    Y_tr_rank   = rank_transform_Y(Y_tr, obs_tr_lag)

    print(f"  [B_rank_lag{lag_num}] per-target top-{TOP_N_PER_TARGET} on rank-Y...")
    exp_b_rank_preds[lag_num] = run_per_target_ridge(
        X_train_aug, X_val_aug, Y_tr_rank,
        label=f"B_rank_lag{lag_num}"
    )

score_b_raw  = official_score(build_full_pred_df(exp_b_raw_preds),  val_solution_raw)
score_b_rank = official_score(build_full_pred_df(exp_b_rank_preds), val_solution_raw)
print(f"\nExp B (per-target top-50, raw-Y):  {score_b_raw:.6f}")
print(f"Exp B (per-target top-50, rank-Y): {score_b_rank:.6f}")

Running Experiment B (per-target top-50) for all 4 lags...

--- Lag 1 ---
  [B_raw_lag1] per-target top-50...
    B_raw_lag1 target 0/106  alpha=100.0
    B_raw_lag1 target 50/106  alpha=100000.0
    B_raw_lag1 target 100/106  alpha=1000.0
  [B_rank_lag1] per-target top-50 on rank-Y...
    B_rank_lag1 target 0/106  alpha=100000.0
    B_rank_lag1 target 50/106  alpha=1000.0
    B_rank_lag1 target 100/106  alpha=100.0

--- Lag 2 ---
  [B_raw_lag2] per-target top-50...
    B_raw_lag2 target 0/106  alpha=0.001
    B_raw_lag2 target 50/106  alpha=1000.0
    B_raw_lag2 target 100/106  alpha=1000.0
  [B_rank_lag2] per-target top-50 on rank-Y...
    B_rank_lag2 target 0/106  alpha=100000.0
    B_rank_lag2 target 50/106  alpha=1000.0
    B_rank_lag2 target 100/106  alpha=100000.0

--- Lag 3 ---
  [B_raw_lag3] per-target top-50...
    B_raw_lag3 target 0/106  alpha=100000.0
    B_raw_lag3 target 50/106  alpha=100000.0
    B_raw_lag3 target 100/106  alpha=1000.0
  [B_rank_lag3] per-target top-50 

## Experiment C — Conservative & PCA Ridge

Two regularisation-diversified Ridge variants, each trained on `X_train_c` only:

| Model | Features | α | Y | Rationale |
|---|---|---|---|---|
| `C_conservative` | All 2850 cols | Fixed 10 000 | Rank | Cross-regime stability via strong regularisation |
| `C_pca50` | Top-50 PCA components | CV-tuned | Rank | Compact orthogonal space; avoids multicollinearity |

Together with A and B they form a 6-model pool for `ensemble_equal`.

In [8]:
# ── Experiment C — Additional Ridge variants for a more robust ensemble ──────
#
# C_conservative : Ridge with fixed α=1e4 (very regularized, rank-Y)
#                  More stable across market regimes than the CV-tuned A models.
# C_pca50        : PCA(50) + CV-tuned Ridge (rank-Y)
#                  Compact orthogonal feature space avoids multicollinearity;
#                  often generalises better than full-width Ridge.
#
# Both train on X_train_c only and predict on X_val_c (392 rows),
# exactly like Experiments A and B.

from sklearn.decomposition import PCA as _PCA


def run_exp_c_conservative(X_tr, X_vl, Y_tr, obs_tr_all, lag_num, alpha=1e4):
    """Very-large-alpha Ridge (rank-Y) — cross-regime regularisation."""
    tgt_names   = list(Y_tr.columns)
    obs_tr_cols = [f"{c}_observed" for c in tgt_names
                   if f"{c}_observed" in obs_tr_all.columns]
    obs_tr_lag  = obs_tr_all[obs_tr_cols] if obs_tr_cols else None
    Y_tr_rank   = rank_transform_Y(Y_tr, obs_tr_lag)

    scaler_X = StandardScaler()
    X_tr_s   = scaler_X.fit_transform(X_tr.values)
    X_vl_s   = scaler_X.transform(X_vl.values)

    scaler_Y = StandardScaler()
    Y_tr_s   = scaler_Y.fit_transform(Y_tr_rank.values)

    model = Ridge(alpha=alpha, random_state=42)
    model.fit(X_tr_s, Y_tr_s)
    return scaler_Y.inverse_transform(model.predict(X_vl_s))   # (n_val, 106)


def run_exp_c_pca(X_tr, X_vl, Y_tr, obs_tr_all, lag_num, n_components=50):
    """PCA(50) + CV-tuned Ridge (rank-Y) — compact orthogonal features."""
    tgt_names   = list(Y_tr.columns)
    obs_tr_cols = [f"{c}_observed" for c in tgt_names
                   if f"{c}_observed" in obs_tr_all.columns]
    obs_tr_lag  = obs_tr_all[obs_tr_cols] if obs_tr_cols else None
    Y_tr_rank   = rank_transform_Y(Y_tr, obs_tr_lag)

    scaler_X = StandardScaler()
    X_tr_s   = scaler_X.fit_transform(X_tr.values)
    X_vl_s   = scaler_X.transform(X_vl.values)

    pca      = _PCA(n_components=n_components, random_state=42)
    X_tr_pca = pca.fit_transform(X_tr_s)
    X_vl_pca = pca.transform(X_vl_s)

    scaler_Y = StandardScaler()
    Y_tr_s   = scaler_Y.fit_transform(Y_tr_rank.values)

    best_alpha = tune_alpha(X_tr_pca, Y_tr_s)
    print(f"    [C_pca50_lag{lag_num}] alpha={best_alpha}")

    model = Ridge(alpha=best_alpha, random_state=42)
    model.fit(X_tr_pca, Y_tr_s)
    return scaler_Y.inverse_transform(model.predict(X_vl_pca))   # (n_val, 106)


print("Running Experiment C (conservative + PCA Ridge) for all 4 lags...")
exp_c_conservative_preds = {}
exp_c_pca_preds          = {}

for lag_num in [1, 2, 3, 4]:
    print(f"\n  --- Lag {lag_num} ---")
    Y_tr = Y_dfs[lag_num]["train"]
    print(f"    [C_conservative_lag{lag_num}] alpha=1e4 (fixed)")
    exp_c_conservative_preds[lag_num] = run_exp_c_conservative(
        X_train_c, X_val_c, Y_tr, obs_train_all, lag_num)
    exp_c_pca_preds[lag_num] = run_exp_c_pca(
        X_train_c, X_val_c, Y_tr, obs_train_all, lag_num)

score_c1 = official_score(build_full_pred_df(exp_c_conservative_preds).iloc[:VAL_TEST_SPLIT],
                          val_solution_raw)
score_c2 = official_score(build_full_pred_df(exp_c_pca_preds).iloc[:VAL_TEST_SPLIT],
                          val_solution_raw)
print(f"\nC_conservative val score: {score_c1:.6f}")
print(f"C_pca50        val score: {score_c2:.6f}")


Running Experiment C (conservative + PCA Ridge) for all 4 lags...

  --- Lag 1 ---
    [C_conservative_lag1] alpha=1e4 (fixed)
    [C_pca50_lag1] alpha=10000.0

  --- Lag 2 ---
    [C_conservative_lag2] alpha=1e4 (fixed)
    [C_pca50_lag2] alpha=1000.0

  --- Lag 3 ---
    [C_conservative_lag3] alpha=1e4 (fixed)
    [C_pca50_lag3] alpha=1000.0

  --- Lag 4 ---
    [C_conservative_lag4] alpha=1e4 (fixed)
    [C_pca50_lag4] alpha=1000.0

C_conservative val score: 0.280793
C_pca50        val score: 0.286845


## Experiment D — Momentum / Lag-Correction Predictor

Inspired by the 6th-place solution: the most impactful trick was **lag correction** —
blending model predictions with the most recent known return signal.
A pure *persistence* baseline (predict = last known return) already captures a
significant portion of the performance, and including it in the ensemble adds a
complementary signal to the Ridge models.

**Signal source:**
- **Val rows (0–257, date_id 1569–1826):** `train_labels.csv` shifted back by `lag_k+1`
  rows (aligned with `aligned` DataFrame already built in Step 3).
- **Test rows (258–391, date_id 1827–1960):** `label_target_*_lag*` columns from
  `test.csv` (released by the competition API at each test date).

No future information is used: the momentum signal at date *d* only sees labels
from dates ≤ *d − 2* (lag offset confirmed earlier).

In [10]:
# ── Experiment D — Momentum / Lag-Correction Predictor ───────────────────────
# For each target at date d, predict = most recently known return of that target.
#
# Val rows  (j=0..257):  train_labels at position (n_train+j - offset) in `aligned`
#                         offset = lag_num+1  (lag1→back 2, lag2→back 3, ...)
# Test rows (j=258..391): X_val_f0's label_target_{i}_lag{k} column (API-released)
#
# For lag groups 2-4, the underlying commodity index is gi % 106.

def build_momentum_predictions():
    """Return (n_val=392, 424) momentum prediction array."""
    result = np.zeros((n_val, 424), dtype=float)

    for lag_num, indices in LAG_GROUPS.items():
        offset   = lag_num + 1          # lag1→2, lag2→3, lag3→4, lag4→5
        src_cols = [f"target_{gi % 106}" for gi in indices]   # 106 label cols

        # --- Val rows: use aligned (train_labels) shifted back by offset ---
        for j in range(VAL_TEST_SPLIT):
            src_row = n_train + j - offset
            if 0 <= src_row < len(aligned):
                vals = aligned[src_cols].iloc[src_row].fillna(0.0).values
                for li, gi in enumerate(indices):
                    result[j, gi] = float(vals[li])

        # --- Test rows: use competition-API lag columns from X_val_f0 ---
        for li, gi in enumerate(indices):
            orig_tgt = gi % 106
            col = f"label_target_{orig_tgt}_lag{lag_num}"
            if col in X_val_raw.columns:
                vals = X_val_f0[col].values[VAL_TEST_SPLIT:]   # 134 test rows
                result[VAL_TEST_SPLIT:, gi] = vals

    return result


print("Building Experiment E — Momentum predictor...")
_momentum_arr = build_momentum_predictions()

exp_d_preds = {}
for lag_num, indices in LAG_GROUPS.items():
    exp_d_preds[lag_num] = _momentum_arr[:, indices]

pred_df_d = build_full_pred_df(exp_d_preds)
score_d_val  = official_score(pred_df_d.iloc[:VAL_TEST_SPLIT], val_solution_raw)
score_d_test = official_score(pred_df_d.iloc[VAL_TEST_SPLIT:], test_ground_truth)
print(f"D_momentum  Val score : {score_d_val:.6f}")
print(f"D_momentum  Test score: {score_d_test:.6f}  (reported in Step 6 only)")


Building Experiment E — Momentum predictor...
E_momentum  Val score : -0.000951
E_momentum  Test score: 0.000000  (reported in Step 6 only)


## Experiment E — HistGradientBoosting Shared Model

Sklearn's `HistGradientBoostingRegressor` is a fast gradient-boosted tree model that
handles missing values natively and requires no external dependencies.
It can capture non-linear feature interactions that Ridge misses.
Conservative settings (shallow trees, high regularisation) limit overfitting.

One `MultiOutputRegressor(HGBR)` is trained per lag group (4 total).
Predictions are generated for all 392 val+test rows.
Training time: ~30–120 seconds total.

In [11]:
# ── Experiment E — HistGradientBoosting model ────────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

print("Running Experiment E — HistGradientBoosting for all 4 lag groups...")

# Use un-scaled X: HGBR is tree-based, doesn't need scaling
X_tr_f = X_train_c.values   # (1549, 2850)
X_vl_f = X_val_c.values     # (392, 2850)

exp_e_preds = {}   # keep same variable name for Cell 22 compatibility

for lag_num, indices in LAG_GROUPS.items():
    print(f"  Lag {lag_num}...", end=" ", flush=True)
    Y_tr   = Y_dfs[lag_num]["train"]
    obs_tr = obs_train_all[[f"target_{i}_observed" for i in indices
                             if f"target_{i}_observed" in obs_train_all.columns]]
    obs_tr_arg = obs_tr if not obs_tr.empty else None
    Y_tr_rank  = rank_transform_Y(Y_tr, obs_tr_arg).values

    _base_hgbr = HistGradientBoostingRegressor(
        max_iter=80,
        max_leaf_nodes=15,
        min_samples_leaf=50,
        learning_rate=0.1,
        l2_regularization=1.0,
        max_features=0.5,
        random_state=42,
    )
    model_f = MultiOutputRegressor(_base_hgbr, n_jobs=-1)
    model_f.fit(X_tr_f, Y_tr_rank)
    exp_e_preds[lag_num] = model_f.predict(X_vl_f)   # (392, 106)
    print("done")

pred_df_e = build_full_pred_df(exp_e_preds)
score_e_val  = official_score(pred_df_e.iloc[:VAL_TEST_SPLIT], val_solution_raw)
score_e_test = official_score(pred_df_e.iloc[VAL_TEST_SPLIT:], test_ground_truth)
print(f"E_hgbr  Val score : {score_e_val:.6f}")
print(f"E_hgbr  Test score: {score_e_test:.6f}  (reported in Step 6 only)")


Running Experiment F — HistGradientBoosting for all 4 lag groups...
  Lag 1... done
  Lag 2... done
  Lag 3... done
  Lag 4... done
F_hgbr  Val score : 0.124955
F_hgbr  Test score: 0.000000  (reported in Step 6 only)


## Step 5 — Ensemble: Candidates & Optimization

Pool all candidate prediction DataFrames into a dictionary, score each
individually on the validation window, then find both equal-weight and
validation-optimized ensemble weights.

All ensemble design decisions use **validation data only** (date_id 1569–1826).
The final model `ensemble_equal` is pre-committed before any test scoring.

In [12]:
# Collect all candidate 424-column prediction DataFrames
candidates = {
    "A_no_aug":       build_full_pred_df(exp_a_preds),
    "A_aug":          build_full_pred_df(exp_a_aug_preds),
    "B_raw":          build_full_pred_df(exp_b_raw_preds),
    "B_rank":         build_full_pred_df(exp_b_rank_preds),
    "C_conservative": build_full_pred_df(exp_c_conservative_preds),
    "C_pca50":        build_full_pred_df(exp_c_pca_preds),
    "D_momentum":     pred_df_d,
}

candidates["E_hgbr"] = pred_df_e


# Score each candidate individually on val
print("\nIndividual candidate val scores:")
print(f"{'Name':<16} {'Val Score':>12}")
print("-" * 30)
ind_scores = {}
for name, pred_df in candidates.items():
    sc = official_score(pred_df.iloc[:VAL_TEST_SPLIT], val_solution_raw)
    ind_scores[name] = sc
    print(f"{name:<16} {sc:>12.6f}")


Notebook-3 best prediction not found — skipping.

Individual candidate val scores:
Name                Val Score
------------------------------
A_no_aug             0.185347
A_aug                0.185347
B_raw                0.231981
B_rank               0.347119
C_conservative       0.280793
C_pca50              0.286845
E_momentum          -0.000951
F_hgbr               0.124955


In [13]:
# Validation-optimized ensemble: grid search on simplex
cand_names  = list(candidates.keys())
cand_arrays = [candidates[n].values for n in cand_names]   # list of (392, 424) arrays
n_cands     = len(cand_names)

def ensemble_score(weights):
    """Negative official score (for minimizer)."""
    weights = np.array(weights)
    weights = np.clip(weights, 0, None)
    w_sum   = weights.sum()
    if w_sum < 1e-12:
        return 0.0
    weights /= w_sum
    combined     = sum(w * a for w, a in zip(weights, cand_arrays))
    pred_df_val  = pd.DataFrame(combined[:VAL_TEST_SPLIT], columns=ALL_TARGET_COLS)
    return -official_score(pred_df_val, val_solution_raw)

# Grid search on coarse grid first (step 0.2)
print("Grid searching ensemble weights (coarse)...")
best_w, best_sc = None, -np.inf

if n_cands == 2:
    grid = [(a, 1 - a) for a in np.arange(0, 1.01, 0.1)]
elif n_cands == 3:
    grid = [(a, b, 1-a-b) for a in np.arange(0,1.01,0.2)
            for b in np.arange(0, 1-a+0.01, 0.2) if a + b <= 1.0]
elif n_cands == 4:
    grid = [(a,b,c,1-a-b-c) for a in np.arange(0,1.01,0.2)
            for b in np.arange(0,1-a+0.01,0.2)
            for c in np.arange(0,1-a-b+0.01,0.2) if a+b+c <= 1.0]
else:
    # For 5+ candidates: equal + individual + pairs of zero-weight
    grid = [tuple([1/n_cands]*n_cands)]
    for k in range(n_cands):         # each model solo
        w = [0.0]*n_cands; w[k] = 1.0; grid.append(tuple(w))
    for k in range(n_cands):         # one model dropped
        w = [1/(n_cands-1)]*n_cands; w[k] = 0.0
        w = [x/sum(w) for x in w]; grid.append(tuple(w))

for w in grid:
    sc = -ensemble_score(list(w))
    if sc > best_sc:
        best_sc, best_w = sc, list(w)

# Refine with scipy minimize (Nelder-Mead)
res = minimize(ensemble_score, best_w, method="Nelder-Mead",
               options={"maxiter": 500, "xatol": 1e-4, "fatol": 1e-4})
refined_w = np.clip(res.x, 0, None)
refined_w /= refined_w.sum()
refined_sc = -ensemble_score(refined_w)

if refined_sc > best_sc:
    best_sc, best_w = refined_sc, list(refined_w)

print(f"\nOptimized ensemble weights:")
for name, w in zip(cand_names, best_w):
    print(f"  {name:<14}: {w:.4f}")
print(f"Optimized ensemble score: {best_sc:.6f}")

# Also compare with simple equal-weight
eq_w  = [1/n_cands]*n_cands
eq_sc = -ensemble_score(eq_w)
print(f"Equal-weight ensemble:    {eq_sc:.6f}")

Grid searching ensemble weights (coarse)...

Optimized ensemble weights:
  A_no_aug      : 0.0000
  A_aug         : 0.0000
  B_raw         : 0.0000
  B_rank        : 0.7975
  C_conservative: 0.0517
  C_pca50       : 0.1490
  E_momentum    : 0.0018
  F_hgbr        : 0.0000
Optimized ensemble score: 0.392803
Equal-weight ensemble:    0.244744


## Step 6 — Model Parameters

Best alpha values selected per model variant during cross-validated training.

In [17]:
# ── Step 6 — Complete parameter summary for all models ──────────────────────
#
# Covers: Exp A, B, C, D, E
# Alpha values are re-derived here (no Ridge fit; alpha search only).
# Fixed/structural parameters are printed directly from constants.
# ─────────────────────────────────────────────────────────────────────────────

SEP = "=" * 62

# ═══════════════════════════════════════════════════════════
# Experiment A — Shared Ridge (rank-Y)
#   Parameters: α tuned by CV Spearman, 1 value per lag group
#   Models: A_no_aug, A_aug  (identical because aug cols = 0)
# ═══════════════════════════════════════════════════════════
print(SEP)
print("Exp A — Shared Ridge (A_no_aug, A_aug)")
print("  Y transform : daily cross-sectional rank  →  [−1, 1]")
print("  Features    : all 2850 base columns")
print("  alpha grid  :", ALPHA_GRID)
print("  CV splits   :", N_SPLITS_CV, " (TimeSeriesSplit, Spearman IC)")
print(SEP)

exp_a_alphas = {}
for lag_num in [1, 2, 3, 4]:
    Y_tr = Y_dfs[lag_num]["train"]
    obs_tr_cols = [f"{c}_observed" for c in Y_tr.columns
                   if f"{c}_observed" in obs_train_all.columns]
    obs_tr_lag  = obs_train_all[obs_tr_cols] if obs_tr_cols else None
    Y_tr_rank   = rank_transform_Y(Y_tr, obs_tr_lag)
    scaler_X = StandardScaler()
    X_s = scaler_X.fit_transform(X_train_c.values)
    scaler_Y = StandardScaler()
    Y_s = scaler_Y.fit_transform(Y_tr_rank.values)
    a = tune_alpha(X_s, Y_s)
    exp_a_alphas[lag_num] = a
    print(f"  Lag {lag_num}  →  alpha = {a}")
print(f"  (A_no_aug and A_aug share the same alpha; aug cols are all-zero)")


# ═══════════════════════════════════════════════════════════
# Experiment B — Per-target Ridge (top-50 features, raw / rank-Y)
#   Parameters: α tuned per target independently (424 × 2 = 848 values)
# ═══════════════════════════════════════════════════════════
print()
print(SEP)
print("Exp B — Per-target Ridge (B_raw, B_rank)")
print("  Feature selection: top-", TOP_N_PER_TARGET,
      "features per target by |Pearson corr| on training window")
print("  Y transform  B_raw : raw (unranked)")
print("  Y transform  B_rank: daily cross-sectional rank")
print("  alpha grid  :", ALPHA_GRID)
print("  CV splits   :", N_SPLITS_CV)
print(SEP)

def _collect_alphas(X_tr, Y_tr_df, top_n=TOP_N_PER_TARGET):
    Xv    = X_tr.values
    X_c   = Xv - Xv.mean(axis=0, keepdims=True)
    x_std = X_c.std(axis=0, ddof=0) + 1e-12
    n     = Xv.shape[0]
    tscv  = TimeSeriesSplit(n_splits=N_SPLITS_CV)
    alphas = []
    for tgt in Y_tr_df.columns:
        y    = Y_tr_df[tgt].values
        y_c  = y - y.mean()
        y_s  = y_c.std(ddof=0) + 1e-12
        corr = (X_c.T @ y_c) / (x_std * y_s * n)
        top_idx = np.argsort(-np.abs(corr))[:top_n]
        sel     = [X_tr.columns[i] for i in top_idx]
        scaler_X = StandardScaler()
        X_s  = scaler_X.fit_transform(X_tr[sel].values)
        scaler_y = StandardScaler()
        y_s_arr  = scaler_y.fit_transform(y.reshape(-1,1)).ravel()
        best_a, best_sc = ALPHA_GRID[0], -np.inf
        for a in ALPHA_GRID:
            fold_sps = []
            for tr_idx, vl_idx in tscv.split(X_s):
                m = Ridge(alpha=a, random_state=42)
                m.fit(X_s[tr_idx], y_s_arr[tr_idx])
                p = m.predict(X_s[vl_idx])
                if np.std(y_s_arr[vl_idx]) > 1e-12 and np.std(p) > 1e-12:
                    fold_sps.append(float(spearmanr(y_s_arr[vl_idx], p).correlation))
            sc = float(np.mean(fold_sps)) if fold_sps else -np.inf
            if sc > best_sc:
                best_sc, best_a = sc, a
        alphas.append(best_a)
    return alphas

exp_b_alphas = {}
for variant, use_rank in [("B_raw", False), ("B_rank", True)]:
    exp_b_alphas[variant] = {}
    all_alphas = []
    print(f"\n  {variant}  ({'rank-Y' if use_rank else 'raw-Y'}):")
    for lag_num in [1, 2, 3, 4]:
        Y_tr = Y_dfs[lag_num]["train"]
        if use_rank:
            obs_tr_cols = [f"{c}_observed" for c in Y_tr.columns
                           if f"{c}_observed" in obs_train_all.columns]
            obs_tr_lag  = obs_train_all[obs_tr_cols] if obs_tr_cols else None
            Y_tr = rank_transform_Y(Y_tr, obs_tr_lag)
        lag_alphas = _collect_alphas(X_train_c, Y_tr)
        exp_b_alphas[variant][lag_num] = lag_alphas
        all_alphas.extend(lag_alphas)
        print(f"    Lag {lag_num} sample  →  "
              f"target_0={lag_alphas[0]:.0f}  "
              f"target_50={lag_alphas[50]:.0f}  "
              f"target_100={lag_alphas[100]:.0f}")
    alpha_arr = np.array(all_alphas)
    counts    = {a: int((alpha_arr == a).sum()) for a in ALPHA_GRID}
    print(f"    Distribution across all 424 targets:")
    for a in ALPHA_GRID:
        cnt = counts[a]
        if cnt == 0:
            continue
        bar = "█" * max(1, int(cnt / max(counts.values()) * 20))
        print(f"      alpha={a:>10.4g}  {cnt:>4} targets  {bar}")
    print(f"    Median={np.median(alpha_arr):.0f}  "
          f"Mode={max(counts, key=counts.get):.0f}  "
          f"Range=[{alpha_arr.min():.4g}, {alpha_arr.max():.4g}]")


# ═══════════════════════════════════════════════════════════
# Experiment C — Conservative & PCA Ridge
# ═══════════════════════════════════════════════════════════
print()
print(SEP)
print("Exp C — Conservative & PCA Ridge")
print(SEP)
print()
print("  C_conservative:")
print("    Y transform : daily cross-sectional rank")
print("    Features    : all 2850 base columns")
print("    alpha       : FIXED = 1e+04  (no CV tuning, high regularisation)")
print()
print("  C_pca50:")
print("    Y transform : daily cross-sectional rank")
print("    Features    : PCA(n_components=50) from 2850 columns")
print("    alpha grid  :", ALPHA_GRID)
print("    CV splits   :", N_SPLITS_CV)

exp_c_pca_alphas = {}
print()
print("  C_pca50 best alpha per lag group:")
for lag_num in [1, 2, 3, 4]:
    Y_tr = Y_dfs[lag_num]["train"]
    obs_tr_cols = [f"{c}_observed" for c in Y_tr.columns
                   if f"{c}_observed" in obs_train_all.columns]
    obs_tr_lag  = obs_train_all[obs_tr_cols] if obs_tr_cols else None
    Y_tr_rank   = rank_transform_Y(Y_tr, obs_tr_lag)
    scaler_X = StandardScaler()
    X_s  = scaler_X.fit_transform(X_train_c.values)
    pca  = PCA(n_components=50, random_state=42)
    X_pca = pca.fit_transform(X_s)
    scaler_Y = StandardScaler()
    Y_s = scaler_Y.fit_transform(Y_tr_rank.values)
    a = tune_alpha(X_pca, Y_s)
    exp_c_pca_alphas[lag_num] = a
    print(f"    Lag {lag_num}  →  alpha = {a}  "
          f"(PCA explains {pca.explained_variance_ratio_.sum()*100:.1f}% variance)")


# ═══════════════════════════════════════════════════════════
# Experiment D — Momentum / Lag-Correction Predictor
# ═══════════════════════════════════════════════════════════
print()
print(SEP)
print("Exp D — Momentum / Lag-Correction Predictor (D_momentum)")
print(SEP)
print("  Algorithm   : Persistence (no training, no alpha)")
print("  Formula     : prediction(date d, target i) = train_labels[target i] at d − (lag_k + 1)")
print("  Lag offsets :")
for lag_num in [1, 2, 3, 4]:
    print(f"    Lag group {lag_num}  →  look back {lag_num + 1} rows  "
          f"(targets {LAG_GROUPS[lag_num][0]}–{LAG_GROUPS[lag_num][-1]})")
print("  Val source  : train_labels.csv shifted back by offset rows")
print("  Test source : label_target_*_lag* columns from test.csv (API-released)")


# ═══════════════════════════════════════════════════════════
# Experiment E — HistGradientBoosting
# ═══════════════════════════════════════════════════════════
print()
print(SEP)
print("Exp E — HistGradientBoosting Shared Model (E_hgbr)")
print(SEP)
print("  Y transform     : daily cross-sectional rank")
print("  Features        : all 2850 base columns (no scaling needed for trees)")
print("  max_iter        : 80   (number of boosting rounds)")
print("  max_leaf_nodes  : 15   (tree depth control)")
print("  min_samples_leaf: 50   (minimum samples per leaf, anti-overfit)")
print("  learning_rate   : 0.10")
print("  l2_regularization: 1.0")
print("  max_features    : 0.5  (50% column subsampling per split)")
print("  random_state    : 42")
print("  n_jobs          : −1   (all cores, via MultiOutputRegressor)")
print("  Models trained  : 4 (one per lag group, each predicting 106 targets)")


# ═══════════════════════════════════════════════════════════
# Save all alpha values to CSV
# ═══════════════════════════════════════════════════════════
rows = []
for lag_num in [1, 2, 3, 4]:
    for j, tgt in enumerate([f"target_{i}" for i in LAG_GROUPS[lag_num]]):
        rows.append({
            "target":          tgt,
            "lag_group":       lag_num,
            "A_alpha":         exp_a_alphas[lag_num],
            "B_raw_alpha":     exp_b_alphas["B_raw"][lag_num][j],
            "B_rank_alpha":    exp_b_alphas["B_rank"][lag_num][j],
            "C_conservative_alpha": "1e4 (fixed)",
            "C_pca50_alpha":   exp_c_pca_alphas[lag_num],
            "D_momentum":      "persistence (no alpha)",
            "E_hgbr":          "max_iter=80, max_leaf_nodes=15 (fixed)",
        })
os.makedirs(OUT_DIR, exist_ok=True)
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/All_Alpha_Parameters.csv", index=False)
print(f"\nSaved: {OUT_DIR}/All_Alpha_Parameters.csv")


Exp A — Shared Ridge (A_no_aug, A_aug)
  Y transform : daily cross-sectional rank  →  [−1, 1]
  Features    : all 2850 base columns
  alpha grid  : [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]
  CV splits   : 5  (TimeSeriesSplit, Spearman IC)
  Lag 1  →  alpha = 1000.0
  Lag 2  →  alpha = 1000.0
  Lag 3  →  alpha = 1000.0
  Lag 4  →  alpha = 1000.0
  (A_no_aug and A_aug share the same alpha; aug cols are all-zero)

Exp B — Per-target Ridge (B_raw, B_rank)
  Feature selection: top- 50 features per target by |Pearson corr| on training window
  Y transform  B_raw : raw (unranked)
  Y transform  B_rank: daily cross-sectional rank
  alpha grid  : [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]
  CV splits   : 5

  B_raw  (raw-Y):
    Lag 1 sample  →  target_0=100  target_50=100000  target_100=1000
    Lag 2 sample  →  target_0=0  target_50=1000  target_100=1000
    Lag 3 sample  →  target_0=100000  target_50=100000  target_100=1000
    Lag 

## Step 7 — Final Offline Test Scoring

Final model is **pre-committed** as `ensemble_equal` (equal-weight average of A_no_aug, A_aug, B_raw, B_rank)
before any test scores are observed.

- **Validation** (date_id 1569–1826, 258 rows): scored against `val_solution_raw` from `train_labels.csv`
- **Final Test** (date_id 1827–1960, 134 rows): scored against `test_ground_truth`

`test_ground_truth` was reconstructed from `test_labels_lag_1~4.csv`, aligned by `label_date_id`.  
This is an **offline evaluation** using released public test files — not an official Kaggle leaderboard score.


In [14]:
# ── Final model pre-committed BEFORE any test scoring ────────────────────────
# Any model selection must use training-period validation only (val_solution_raw).
# test_ground_truth is used solely for reporting the final offline result here.

final_model_name = "ensemble_equal"   # pre-committed, not chosen by test score

comb_opt = sum(w * a for w, a in zip(best_w, cand_arrays))   # (392, 424)
comb_eq  = sum(w * a for w, a in zip(eq_w,   cand_arrays))   # (392, 424)

all_preds = {
    **candidates,
    "ensemble_optimized": pd.DataFrame(comb_opt, columns=ALL_TARGET_COLS),
    "ensemble_equal":     pd.DataFrame(comb_eq,  columns=ALL_TARGET_COLS),
}

# Score each model on validation window (258 rows) and final test window (134 rows)
rows = []
print(f"\n{'─'*70}")
print(f"{'Model':<24} {'Val (258 rows)':>16} {'Test (134 rows)':>16}")
print(f"{'─'*70}")
for name, pred_df in all_preds.items():
    sc_val  = official_score(pred_df.iloc[:VAL_TEST_SPLIT].reset_index(drop=True),
                             val_solution_raw)
    sc_test = official_score(pred_df.iloc[VAL_TEST_SPLIT:].reset_index(drop=True),
                             test_ground_truth)
    rows.append((name, sc_val, sc_test))
    marker = "  ← FINAL SUBMISSION" if name == final_model_name else ""
    print(f"  {name:<22} {sc_val:>16.6f} {sc_test:>16.6f}{marker}")
print(f"{'─'*70}")

final_pred_df    = all_preds[final_model_name].copy()
final_val_score  = official_score(final_pred_df.iloc[:VAL_TEST_SPLIT].reset_index(drop=True),
                                  val_solution_raw)
final_test_score = official_score(final_pred_df.iloc[VAL_TEST_SPLIT:].reset_index(drop=True),
                                  test_ground_truth)

print(f"\nFinal model: {final_model_name}")
print(f"  Validation score   (date_id 1569\u20131826, 258 rows): {final_val_score:.6f}")
print(f"  Offline test score (date_id 1827\u20131960, 134 rows): {final_test_score:.6f}")
print()
# Ensemble formula
n_eq = len(cand_names)
formula_terms = " + ".join(f"1/{n_eq} \u00d7 {name}" for name in cand_names)
print(f"  {final_model_name}  =  {formula_terms}")
print(f"\n[NOTE] Offline evaluation using released public test files.")
print(f"       Test labels reconstructed from test_labels_lag_1~4.csv by label_date_id alignment.")
print(f"       Not an official Kaggle leaderboard score.")

# ── Save outputs ──────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
final_pred_df.to_csv(f"{OUT_DIR}/output_Ridge_RankY_final.csv", index=False)

score_summary = pd.DataFrame(rows, columns=["name", "val_258_score", "test_134_score"])
score_summary.to_csv(f"{OUT_DIR}/Final_Score_Summary.csv", index=False)

pd.DataFrame({"candidate": cand_names,
              "optimized_weight": best_w,
              "equal_weight":     eq_w}).to_csv(f"{OUT_DIR}/Ensemble_Weights.csv", index=False)

print(f"\nSaved: output_Ridge_RankY_final.csv")
print(f"Saved: Final_Score_Summary.csv")
print(f"Saved: Ensemble_Weights.csv")



──────────────────────────────────────────────────────────────────────
Model                      Val (258 rows)  Test (134 rows)
──────────────────────────────────────────────────────────────────────
  A_no_aug                       0.185347         0.224590
  A_aug                          0.185347         0.224590
  B_raw                          0.231981         0.051013
  B_rank                         0.347119         0.106978
  C_conservative                 0.280793         0.315748
  C_pca50                        0.286845         0.143220
  E_momentum                    -0.000951        -0.163262
  F_hgbr                         0.124955         0.246965
  ensemble_optimized             0.392803         0.189018
  ensemble_equal                 0.244744         0.281504  ← FINAL SUBMISSION
──────────────────────────────────────────────────────────────────────

Final model: ensemble_equal
  Validation score   (date_id 1569–1826, 258 rows): 0.244744
  Offline test score (date_